# Drum RM – copaci fără marcare (model care caută după nume)

Un model gata antrenat (YOLO-World) caută în poze ce îi scriem: **trunchi de copac**, **groapă**, **indicator rutier**.
Nu trebuie marcat nimic.

1. **Runtime → Change runtime type → T4 GPU**.
2. **Runtime → Run all**.
3. Când se cere, încarci câteva poze sau cadre cu copaci (le poți selecta pe toate odată).
4. Vezi pozele cu chenarele găsite. Dacă sunt bune, la final se descarcă **drum_v2.onnx** – îl trimiți lui Claude sau îl pui în folderul `model` al programului Drum.

In [ ]:
!pip -q install ultralytics
import torch; print('Placa video:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NU – alege T4 GPU din Runtime')

In [ ]:
# Modelul care caută după nume. Ordinea claselor e cea din programul Drum: groapa, copac, semn
from ultralytics import YOLOWorld
CLASE = ['pothole', 'tree trunk', 'traffic sign']
m = YOLOWorld('yolov8l-worldv2.pt')
m.set_classes(CLASE)
print('model gata')

In [ ]:
from google.colab import files
import os
os.makedirs('/content/poze', exist_ok=True)
up = files.upload()
for n, b in up.items(): open('/content/poze/' + n, 'wb').write(b)
print(len(up), 'poze încărcate')

In [ ]:
# Pragul de încredere: mai mic = mai mulți copaci găsiți (și mai multe greșeli)
PRAG = 0.10
import glob, cv2
from IPython.display import Image, display
for p in sorted(glob.glob('/content/poze/*')):
    r = m.predict(p, conf=PRAG, imgsz=1280, verbose=False)[0]
    img = cv2.imread(p); n = 0
    for (x0, y0, x1, y1), c, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        col = [(50, 50, 230), (60, 200, 60), (230, 120, 40)][int(c)]
        cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), col, 2)
        if int(c) == 1: cv2.circle(img, (int((x0 + x1) / 2), int(y1)), 6, (0, 0, 255), -1); n += 1
    out = '/content/rez_' + os.path.basename(p).rsplit('.', 1)[0] + '.jpg'; cv2.imwrite(out, img)
    print(os.path.basename(p), ':', n, 'trunchiuri'); display(Image(out, width=900))

In [ ]:
# Modelul pentru programul Drum (merge pe orice calculator, fără placă video)
import shutil
f = m.export(format='onnx', imgsz=640, simplify=True)
shutil.copy(f, '/content/drum_v2.onnx')
open('/content/labels.txt', 'w').write('groapa\ncopac\nsemn\n')
files.download('/content/drum_v2.onnx')